<img src='https://repository-images.githubusercontent.com/121802384/c355bb80-7d42-11e9-9e0e-4729609f9fbc' alt='WRF-Hydro Logo' width="15%"/>

# Lesson 5 - Land surface experiments

## Overview
In [Lesson 4](Lesson-4-run-options.ipynb) we experimented with different initial and boundary conditions (precipitation). 

In this lesson, we experiment with parameter adjustments to evaluate impacts on streamflow.

**NOTE: If you have not completed Lessons 1 through 4, please stop and do so now.**

### Compiling WRF-Hydro
We do not need to modify the code binary to conduct physics experiments, so we will use the previously compiled code.  

If you need to recompile the code, refer to the instructions in [Lesson 2](Lesson-2-run.ipynb).

## Experiment with Modified Parameters
There are a number of key parameters that impact water partitioning, storage, and movement through the model system. We have pulled many of the most important model parameters into NetCDF files to ease parameter display and manipulation, as well as to allow the parameters to vary independently in space. Key terrain routing parameter files include:

| Filename | Description|
| ------------- | ------------- |
| soil_properties.nc | NoahMP soil and vegetation properties (LSM grid) |
| hydro2dtbl.nc | Lateral routing model soil and surface parameters (LSM grid) |
| Fulldom_hires.nc | Lateral routing model high-res parameters (routing grid) |
| GWBUCKPARM.nc | Groundwater baseflow bucket model parameters (groundwater basin objects) |


In this lesson, we will manipulate parameters in the `soil_properties.nc` (*refkdt*, *dksat*, *bexp*) and `Fulldom_hires.nc` (*LKSATFAC*, *RETDEPRTFAC*) files.

### Step 1: Create a new template directory and run default parameter test case
As in the first section, we will make a new simulation directory and use this as a template for creating multiple new simulation directories.

**Step 1a: Create a new template directory for the parameter experiments.**

In [ ]:
%%bash
# Make a new directory for our default baseline simulation
mkdir -p ~/wrf_hydro_training/output/lesson5/run_default_template

# Copy our model files to the simulation directory
cp ~/wrf_hydro_training/wrf_hydro_nwm_public/build/Run/*.TBL \
  ~/wrf_hydro_training/output/lesson5/run_default_template
cp ~/wrf_hydro_training/wrf_hydro_nwm_public/build/Run/wrf_hydro \
  ~/wrf_hydro_training/output/lesson5/run_default_template

# Create symbolic links to files we will not edit
ln -sf ~/wrf_hydro_training/example_case/FORCING \
  ~/wrf_hydro_training/output/lesson5/run_default_template
ln -sf ~/wrf_hydro_training/example_case/DefaultParams/RESTART \
  ~/wrf_hydro_training/output/lesson5/run_default_template/RESTART

# Copy the domain/parameter files so we can modify them
cp -r ~/wrf_hydro_training/example_case/DefaultParams/DOMAIN \
  ~/wrf_hydro_training/output/lesson5/run_default_template/DOMAIN

# Copy namelist files
cp ~/wrf_hydro_training/example_case/DefaultParams/namelist.hrldas \
  ~/wrf_hydro_training/output/lesson5/run_default_template
cp ~/wrf_hydro_training/example_case/DefaultParams/hydro.namelist \
  ~/wrf_hydro_training/output/lesson5/run_default_template

**Step 1b: Setup and run the default parameter baseline experiment**

We now make a new run directory to run the baseline simulation with default (uncalibrated) parameters.

In [ ]:
%%bash
cp -r ~/wrf_hydro_training/output/lesson5/run_default_template \
  ~/wrf_hydro_training/output/lesson5/run_parameter_baseline

Launch the baseline run and make sure it completes successfully.

In [ ]:
%%bash
cd ~/wrf_hydro_training/output/lesson5/run_parameter_baseline
mpirun -np 2 ./wrf_hydro &> run.log

In [ ]:
%%bash
tail -1 ~/wrf_hydro_training/output/lesson5/run_parameter_baseline/diag_hydro.00000

**Step 1c: Plot the hydrograph for the baseline (default) run**

We want to take a quick look at the hydrograph for the default parameter run so we see what behavior we might want to adjust.

We will use Python and the `xarray` library to load the data and plot hydrographs. For an intro to these tools, please see [Lesson 3](Lesson-3-visualize.ipynb).

**Load the xarray python package**

In [ ]:
# Load the required libraries
import xarray as xr
import matplotlib.pyplot as plt
import pandas as pd
import datetime as dt

xr.set_options(display_style="html")
from pandas.plotting import register_matplotlib_converters
register_matplotlib_converters()

In [ ]:
# Pull the channel output files into xarray objects 
chanobs_control = xr.open_mfdataset('~/wrf_hydro_training/output/lesson5/run_parameter_baseline/*CHANOBS*',
                            combine='by_coords')
obs = pd.read_csv('~/wrf_hydro_training/example_case/USGS_obs_2019_2023.csv',dtype=str)
obs['dateTime'] = pd.to_datetime(obs['dateTime'])
obs['streamflow_cms'] = pd.to_numeric(obs['streamflow_cms'])
obs = obs.sort_values(by='dateTime')

**Plot the hydrograph for the headwater gage (feature_id = 6189624, USGS ID 01362200)**

Table 1. USGS Stream Gage IDs and associated WRF-Hydro feature_id indices.

|USGS Gage ID|WRF-Hydro gage feature_id|Gage Information|
|------|:-:|---|
| 01362200 | 6189624 |https://waterdata.usgs.gov/pa/nwis/uv/?site_no=01362200|
| 01362500 | 6189754 |https://waterdata.usgs.gov/nwis/nwismap/?site_no=01362500|
| 01363556 | 6191738 |https://waterdata.usgs.gov/nwis/nwismap/?site_no=01363556|

In [ ]:
# Specify our evaluation gage and associated reach ID
gageid = "01362200"
fid = 6189624

In [ ]:
# Plot the baseline hydrograph
fig, axes = plt.subplots(ncols=1,figsize=(12, 6))
plt.suptitle('Hydrographs for parameter experiment',fontsize=24)
chanobs_control.sel(feature_id = fid).streamflow.plot(label='Control',
                                                        color='red',
                                                        linestyle='-')
obs[obs['site_no'] == gageid].plot(x='dateTime',
                                       y='streamflow_cms',
                                       ax=axes,
                                       label='Observed',
                                       color='grey')
plt.xlim([dt.date(2023, 3, 1), dt.date(2023, 6, 1)])
plt.ylim(0,50)
plt.legend()
plt.show()

### Step 2: Modify NoahMP parameters using NCO tools

We will create a new simulation directory for our parameter manipulation experiment. 

All of the parameter file edits will be done using NCO. NCO (NetCDF Operators, http://nco.sourceforge.net/) is a set of useful utilities to manipulate NetCDF files.

**Step 2a: Setup the parameter experiment run directory**

First make a new run directory where we can start modifying parameter files.

In [ ]:
%%bash
cp -r ~/wrf_hydro_training/output/lesson5/run_default_template \
  ~/wrf_hydro_training/output/lesson5/run_parameter_mods

**Step 2b: Use the NCO command `ncap2` to modify the Xinanjian infiltration scheme parameter values in the `soil_properties.nc` file.**

**BXAJ, AXAJ, XXAJ**

NWMv3.0 uses an infiltration scheme from the Xinanjiang model as implemented in Noah-MP. The Xinanjiang infiltration scheme has 3 free parameters:

- BXAJ - Tension water distribution shape parameter (range: 0-10). B measures the spatial heterogeneity or non-uniformity of the soil moisture storage capacity across the catchment, where lower values are more uniform and higher values are more heterogeneous.
- AXAJ - Tension water distribution inflection parameter (range: -0.49-0.49).
- XXAJ - Free water distribution shape parameter (range: 0-10).

Noah-MP implementation based on:
Knoben, W. J., et al., (2019): Modular Assessment of Rainfall-Runoff Models  Toolbox (MARRMoT) v1.2 an open-source, extendable framework providing implementations  of 46 conceptual hydrologic models as continuous state-space formulations.

We will modify the *BXAJ* and *XXAJ* values, but first we check the current parameter values using `ncdump`.

In [ ]:
%%bash

ncdump -v BXAJ ~/wrf_hydro_training/output/lesson5/run_parameter_mods/DOMAIN/soil_properties.nc \
  | tail -n 10

Note that this parameter's initial values vary in space based on soil texture class. We modify the parameter values using Python to increase by a factor of 7 while maintaining the spatial distribution.

In [ ]:
%%bash

ncap2 -O -s "BXAJ=float(BXAJ*7.0)" \
~/wrf_hydro_training/output/lesson5/run_parameter_mods/DOMAIN/soil_properties.nc \
~/wrf_hydro_training/output/lesson5/run_parameter_mods/DOMAIN/soil_properties.nc

We check to make sure the changes are as expected.

In [ ]:
%%bash

ncdump -v BXAJ ~/wrf_hydro_training/output/lesson5/run_parameter_mods/DOMAIN/soil_properties.nc \
  | tail -n 10

We'll now modify the other Xinanjiang parameters in a similar fashion, increasing the *AXAJ* parameter by a factor of 3 and the *XXAJ* parameter by a factor of 4.

In [ ]:
%%bash

ncdump -v XXAJ ~/wrf_hydro_training/output/lesson5/run_parameter_mods/DOMAIN/soil_properties.nc \
| tail -n 10

In [ ]:
%%bash

ncap2 -O -s "XXAJ=float(XXAJ*4)" \
~/wrf_hydro_training/output/lesson5/run_parameter_mods/DOMAIN/soil_properties.nc \
~/wrf_hydro_training/output/lesson5/run_parameter_mods/DOMAIN/soil_properties.nc

ncdump -v XXAJ ~/wrf_hydro_training/output/lesson5/run_parameter_mods/DOMAIN/soil_properties.nc \
| tail -n 10

And now we adjust AXAJ

In [ ]:
%%bash

ncap2 -O -s "AXAJ=float(AXAJ*3.0)" \
~/wrf_hydro_training/output/lesson5/run_parameter_mods/DOMAIN/soil_properties.nc \
~/wrf_hydro_training/output/lesson5/run_parameter_mods/DOMAIN/soil_properties.nc
ncdump -v AXAJ ~/wrf_hydro_training/output/lesson5/run_parameter_mods/DOMAIN/soil_properties.nc \
| tail -n 10

**Step 2c: Use the NCO command `ncap2` to modify the *dksat* parameter values in the `soil_properties.nc` file.**

**DKSAT and BEXP**

As with most physically-based hydrological models, the soil saturated hydraulic conductivity (*dksat*) controls the speed at which water moves through the subsurface. This is a sensitive parameter in the model, and while easy to measure at the point scale, *dksat* is tricky to estimate at the scale of kilometers. Initial values are estimated based on soil texture class, but reported ranges have large (many orders of magnitude) variability. This is a common calibration parameter, along with the related *bexp* parameter that controls how actual conductivity is scaled from saturated conductivity based on soil water content.

Here we will use the NCO command `ncap2` to increase the *dksat* value by a factor of 2 and *bexp* by a factor of 1.5.

First, we check the current parameter values using `ncdump`.

In [ ]:
%%bash

ncdump -v dksat ~/wrf_hydro_training/output/lesson5/run_parameter_mods/DOMAIN/soil_properties.nc \
| tail -n 10

Note that this parameter's initial values vary in space based on soil texture class. We modify the parameter values using Python to increase by a factor of 2. This maintains the spatial distribution, but encourage faster soil water movement overall.

In [ ]:
%%bash

ncap2 -O -s "dksat=float(dksat*2.0)" \
~/wrf_hydro_training/output/lesson5/run_parameter_mods/DOMAIN/soil_properties.nc \
~/wrf_hydro_training/output/lesson5/run_parameter_mods/DOMAIN/soil_properties.nc
ncdump -v dksat ~/wrf_hydro_training/output/lesson5/run_parameter_mods/DOMAIN/soil_properties.nc \
| tail -n 10

Use `ncdump` to check the starting values of *bexp*. Note that these also vary spatially by soil type. We will increase these values by a factor of 1.5.

In [ ]:
%%bash

ncdump -v bexp ~/wrf_hydro_training/output/lesson5/run_parameter_mods/DOMAIN/soil_properties.nc \
| tail -n 10

In [ ]:
%%bash

ncap2 -O -s "bexp=float(bexp*1.5)" \
~/wrf_hydro_training/output/lesson5/run_parameter_mods/DOMAIN/soil_properties.nc \
~/wrf_hydro_training/output/lesson5/run_parameter_mods/DOMAIN/soil_properties.nc
ncdump -v bexp ~/wrf_hydro_training/output/lesson5/run_parameter_mods/DOMAIN/soil_properties.nc \
| tail -n 10

**SMCMAX**

Soil water holding capacity is controlled by the porosity (*smcmax*), field capacity (*smcref*), and wilting point (*smcwlt*). These properties are specified by soil texture class, but also vary significantly in space so can be calibrated.

We will reduce the maximum soil water pore space (*smcmax*) by a factor of 0.9.

In [ ]:
%%bash

ncdump -v smcmax ~/wrf_hydro_training/output/lesson5/run_parameter_mods/DOMAIN/soil_properties.nc \
| tail -n 10

In [ ]:
%%bash

ncap2 -O -s "smcmax=float(smcmax*0.9)" \
~/wrf_hydro_training/output/lesson5/run_parameter_mods/DOMAIN/soil_properties.nc \
~/wrf_hydro_training/output/lesson5/run_parameter_mods/DOMAIN/soil_properties.nc
ncdump -v smcmax ~/wrf_hydro_training/output/lesson5/run_parameter_mods/DOMAIN/soil_properties.nc \
| tail -n 10

**SLOPE**

There are a number of Noah/NoahMP model parameters that affect water partitioning. One important parameter that we commonly adjust is *slope*. Originally estimated based on land surface topography (hence the name *slope*), the *slope* parameter controls how open or closed the bottom boundary of the soil column is. It is a multiplier on the bottom boundary conductivity. Values range from 0 to 1, where 0 is a completely closed bottom boundary and 1 is completely open. Lower *slope* values will keep more water in the soil column, while higher values will allow more water to drain to the channel or to deeper baseflow stores, depending on the selected baseflow physics options.

In [ ]:
%%bash

ncdump -v slope ~/wrf_hydro_training/output/lesson5/run_parameter_mods/DOMAIN/soil_properties.nc \
| tail -n 10

In [ ]:
%%bash

ncap2 -O -s "slope=float(slope*0+0.01)" \
~/wrf_hydro_training/output/lesson5/run_parameter_mods/DOMAIN/soil_properties.nc \
~/wrf_hydro_training/output/lesson5/run_parameter_mods/DOMAIN/soil_properties.nc
ncdump -v slope ~/wrf_hydro_training/output/lesson5/run_parameter_mods/DOMAIN/soil_properties.nc \
| tail -n 10

**MFSNO**

One of the main Noah-MP snow model parameters is *mfsno*. *mfsno* controls the shape of the snow depletion curve - the relationship between snow depth and snow cover fraction. This relationship affects the energy balance of the snowpack, and therefore snowmelt. *mfsno* is often varied by scale (generally a larger value for a larger scale; Niu &
Yang, 2007) or vegetation type.

In [ ]:
%%bash

ncdump -v mfsno ~/wrf_hydro_training/output/lesson5/run_parameter_mods/DOMAIN/soil_properties.nc \
| tail -n 10

In [ ]:
%%bash

ncap2 -O -s "mfsno=float(mfsno*0+1.0)" \
~/wrf_hydro_training/output/lesson5/run_parameter_mods/DOMAIN/soil_properties.nc \
~/wrf_hydro_training/output/lesson5/run_parameter_mods/DOMAIN/soil_properties.nc
ncdump -v mfsno ~/wrf_hydro_training/output/lesson5/run_parameter_mods/DOMAIN/soil_properties.nc \
| tail -n 10

In [ ]:
%%bash

ncap2 -O -s "vcmx25=float(vcmx25*0.7)" \
~/wrf_hydro_training/output/lesson5/run_parameter_mods/DOMAIN/soil_properties.nc \
~/wrf_hydro_training/output/lesson5/run_parameter_mods/DOMAIN/soil_properties.nc
ncdump -v vcmx25 ~/wrf_hydro_training/output/lesson5/run_parameter_mods/DOMAIN/soil_properties.nc \
| tail -n 10

In [ ]:
%%bash

ncap2 -O -s "cwpvt=float(cwpvt*1.9)" \
~/wrf_hydro_training/output/lesson5/run_parameter_mods/DOMAIN/soil_properties.nc \
~/wrf_hydro_training/output/lesson5/run_parameter_mods/DOMAIN/soil_properties.nc
ncdump -v cwpvt ~/wrf_hydro_training/output/lesson5/run_parameter_mods/DOMAIN/soil_properties.nc \
| tail -n 10

In [ ]:
%%bash

ncap2 -O -s "rsurfsnow=float(rsurfsnow*0+30)" \
~/wrf_hydro_training/output/lesson5/run_parameter_mods/DOMAIN/soil_properties.nc \
~/wrf_hydro_training/output/lesson5/run_parameter_mods/DOMAIN/soil_properties.nc
ncdump -v rsurfsnow ~/wrf_hydro_training/output/lesson5/run_parameter_mods/DOMAIN/soil_properties.nc \
| tail -n 10

In [ ]:
%%bash

ncap2 -O -s "mp=float(mp*0.9)" \
~/wrf_hydro_training/output/lesson5/run_parameter_mods/DOMAIN/soil_properties.nc \
~/wrf_hydro_training/output/lesson5/run_parameter_mods/DOMAIN/soil_properties.nc
ncdump -v mp ~/wrf_hydro_training/output/lesson5/run_parameter_mods/DOMAIN/soil_properties.nc \
| tail -n 10

### Step 3: Modify terrain routing parameters using NCO tools

Now we will turn our attention to the parameters that affect lateral routing behavior.

**Step 3a: Use the NCO command `ncap2` to modify the *LKSATFAC* and *RETDEPRTFAC* parameter values in the `Fulldom_hires.nc` file**

**LKSATFAC**

The Fulldom_hires.nc file contains two parameters that are also important for lateral flow processes. The *LKSATFAC* parameter is a multiplier on the prescribed lateral saturated hydraulic conductivity values specified in `hydro2dtbl.nc`. By default, lateral conductivity in `hydro2dtbl.nc` matches vertical conductivity specified in `soil_properties.nc`. However, in the real world we frequently see many orders of magnitude higher conductivities in the lateral direction vs. the vertical direction (due to soil stratigraphy, preferential flowpaths caused by roots and animals, etc.). *LKSATFAC* is an easy way to adjust this anisotropy, and by default it is set to 1,000.

We will use the NCO command `ncap2` to increase the *LKSATFAC* value to 2,000 (this encompasses the dksat multipler of 2 and the original anisotropy factor of 1,000).

First, we check the current parameter values.

In [ ]:
%%bash

ncdump -v LKSATFAC ~/wrf_hydro_training/output/lesson5/run_parameter_mods/DOMAIN/Fulldom_hires.nc \
| tail -n 10

Then, we modify the parameter values and confirm our changes.

In [ ]:
%%bash

ncap2 -O -s "LKSATFAC=float(LKSATFAC*0.0+2500.0)" \
~/wrf_hydro_training/output/lesson5/run_parameter_mods/DOMAIN/Fulldom_hires.nc \
~/wrf_hydro_training/output/lesson5/run_parameter_mods/DOMAIN/Fulldom_hires.nc
ncdump -v LKSATFAC ~/wrf_hydro_training/output/lesson5/run_parameter_mods/DOMAIN/Fulldom_hires.nc \
| tail -n 10

**RETDEPRTFAC**

The *RETDEPRTFAC* parameter is a multiplier on the maximum retention depth. Ponded water on the surface above this retention depth threshold can be moved around the landscape via overland flow. The default value in the code is quite small (~0.001mm, though variable by terrain slope) to allow almost all ponded water to be available for routing. However, in many regions landscape features like wetlands, small detention ponds, and heavy vegetation litter/debris can trap water on the land surface. Increasing the *RETDEPRTFAC* multiplier will hold more ponded water on the surface before it becomes runoff.

We will use the NCO command `ncap2` to increase the *RETDEPRTFAC* value to 300 (max of 0.3mm).

First, we check the current parameter values.

In [ ]:
%%bash

ncdump -v RETDEPRTFAC ~/wrf_hydro_training/output/lesson5/run_parameter_mods/DOMAIN/Fulldom_hires.nc \
| tail -n 10

Then, we modify the parameter values using Python and confirm our changes.

In [ ]:
%%bash

ncap2 -O -s "RETDEPRTFAC=float(RETDEPRTFAC*0.0+300.0)" \
~/wrf_hydro_training/output/lesson5/run_parameter_mods/DOMAIN/Fulldom_hires.nc \
~/wrf_hydro_training/output/lesson5/run_parameter_mods/DOMAIN/Fulldom_hires.nc
ncdump -v RETDEPRTFAC ~/wrf_hydro_training/output/lesson5/run_parameter_mods/DOMAIN/Fulldom_hires.nc \
| tail -n 10

**GW Bucket Parameters**

The conceptual groundwater "buckets" include tuneable parameters for maximum bucket storage depth (*Zmax*, in mm), the exponent that controls how slowly/quickly the water drains from the buckets into the stream reaches (*Expon*, dimensionless), and a scaling coefficient that controls how much water drains (*Coeff*, dimensionless in this configuration). There is also an optional loss parameter (*Loss*, dimensionless fraction from 0 to 1) that allows drainage from the buckets into a deeper aquifer, however this option should be used cautiously as this water is lost completely from the system so will not conserve mass.

We will use the NCO command `ncap2` to change the *Zmax* and *Expon* parameters. We do not have any prior information about these parameters, so initial values are uniform across the domain. For simplicity, we will replace the values uniformly, though they likely vary in space.

First, we check the current parameter values.

In [ ]:
# Open the groundwater parameter file and select a representative basin that matches 
# the gaged stream reach to look at parameter values.
gwparm = xr.open_dataset("~/wrf_hydro_training/output/lesson5/run_parameter_mods/DOMAIN/GWBUCKPARM.nc")
print(gwparm.where(gwparm['ComID'] == fid, drop=True))

Let's increase the *Zmax* value to 200 mm. We will apply this change everywhere, although storage likely varies in space.

In [ ]:
%%bash

ncap2 -O -s "Zmax=float(Zmax*0.0+200.0)" \
~/wrf_hydro_training/output/lesson5/run_parameter_mods/DOMAIN/GWBUCKPARM.nc \
~/wrf_hydro_training/output/lesson5/run_parameter_mods/DOMAIN/GWBUCKPARM.nc
ncdump -v Zmax ~/wrf_hydro_training/output/lesson5/run_parameter_mods/DOMAIN/GWBUCKPARM.nc \
| tail -n 10

Additionally, let's reduce the *Expon* to 1.0.

In [ ]:
%%bash

ncap2 -O -s "Expon=float(Expon*0.0+1.0)" \
~/wrf_hydro_training/output/lesson5/run_parameter_mods/DOMAIN/GWBUCKPARM.nc \
~/wrf_hydro_training/output/lesson5/run_parameter_mods/DOMAIN/GWBUCKPARM.nc
ncdump -v Expon ~/wrf_hydro_training/output/lesson5/run_parameter_mods/DOMAIN/GWBUCKPARM.nc \
| tail -n 10

**Step 3b: Run the simulation**

Now we are ready to run a simulation with our new modified parameters.

In [ ]:
%%bash
cd ~/wrf_hydro_training/output/lesson5/run_parameter_mods
mpirun -np 2 ./wrf_hydro &> run.log

Check to make sure your run completed successfully.

In [ ]:
%%bash

tail -1 ~/wrf_hydro_training/output/lesson5/run_parameter_mods/diag_hydro.00000

### Results
We will now look at the differences in streamflow between our baseline run with default parmeters and the simulation using our new modified parameters.

We will use Python and the `xarray` library to load the data and plot hydrographs. For an intro to these tools, please see [Lesson 3](Lesson-3-visualize.ipynb).

**Load the CHANOBS streamflow datasets**

We are going to use the CHANOBS files because it will limit the number of grid cells to only those which we have specified have a gage.

In [ ]:
# Pull the channel output files into xarray objects 
chanobs_param_mods = xr.open_mfdataset('~/wrf_hydro_training/output/lesson5/run_parameter_mods/*CHANOBS*',
                                       combine='by_coords')

**Plot the hydrographs**

In [ ]:
fig, axes = plt.subplots(ncols=1,figsize=(12, 6))
plt.suptitle('Hydrographs for parameter experiment',fontsize=24)
chanobs_control.sel(feature_id = fid).streamflow.plot(label='Control',
                                                    color='red',
                                                    linestyle='-')
chanobs_param_mods.sel(feature_id = fid).streamflow.plot(label='Parameter Modifications',
                                                       color='blue',
                                                       linestyle='--')
obs[obs['site_no'] == gageid].plot(x='dateTime',
                                       y='streamflow_cms',
                                       ax=axes,
                                       label='Observed',
                                       color='grey')
plt.xlim([dt.date(2023, 3, 1), dt.date(2023, 6, 1)])
plt.ylim(0,40)
plt.legend()
plt.show()

We see definite differences between the streamflow responses with and without parameter modifications (red solid and black dashed lines, respectively). The low streamflow bias in the original run (black) is mitigated in the "calibrated" run (red), and we now see a slight high bias. So where did all the new streamflow come from?

**Load the LDASOUT land model output files**

In our simulations, we specified land surface model output (LDASOUT) files once per day. We will read those in using xarray.

In [ ]:
ldasout_baseline = xr.open_mfdataset('~/wrf_hydro_training/output/lesson5/run_parameter_baseline/*.LDASOUT*',
                            combine='by_coords')
ldasout_mods = xr.open_mfdataset('~/wrf_hydro_training/output/lesson5/run_parameter_mods/*.LDASOUT*',
                            combine='by_coords')

**Plot the mean soil moisture time series**

In [ ]:
# Calculate the mean bottom-layer soil moisture across the domain
smois_baseline_avg = ldasout_baseline.SOIL_M.sel(soil_layers_stag = 3).mean(dim=('y','x'))
smois_mods_avg = ldasout_mods.SOIL_M.sel(soil_layers_stag = 3).mean(dim=('y','x')) 

# Plot the soil moisture time series
fig, axes = plt.subplots(ncols=1,figsize=(12, 6))
plt.suptitle('Average Soil Moisture: Bottom Layer',fontsize=24)
smois_baseline_avg.plot(label='Control', color='black', linestyle='--')
smois_mods_avg.plot(label='Parameter Modifications', color='red', linestyle='-')
plt.legend()
plt.show()

In [ ]:
# Calculate the mean SWE across the domain
swe_baseline_avg = ldasout_baseline.SNEQV.mean(dim=('y','x'))
swe_mods_avg = ldasout_mods.SNEQV.mean(dim=('y','x')) 

# Plot the SWE time series
fig, axes = plt.subplots(ncols=1,figsize=(12, 6))
plt.suptitle('Average SWE',fontsize=24)
swe_baseline_avg.plot(label='Control', color='black', linestyle='--')
swe_mods_avg.plot(label='Parameter Modifications', color='red', linestyle='-')
plt.legend()
plt.show()

In [ ]:
# Calculate the mean ET across the domain
et_baseline_avg = ldasout_baseline.ACCET.mean(dim=('y','x'))
et_mods_avg = ldasout_mods.ACCET.mean(dim=('y','x')) 

# Plot the SWE time series
fig, axes = plt.subplots(ncols=1,figsize=(12, 6))
plt.suptitle('Average Accumulated ET',fontsize=24)
et_baseline_avg.plot(label='Control', color='black', linestyle='--')
et_mods_avg.plot(label='Parameter Modifications', color='red', linestyle='-')
plt.legend()
plt.show()

**IMPORTANT NOTE**: We abbreviated this parameter calibration exercise to fit within a short lesson. In practice, you would want to give the model time to adjust to a new parameter set by running an extended "spin-up" period before the time period you are evaluating. You would also want to expose the model to a wider range of conditions than the short event demostrated here. Model parameters calibrated to a short event may not transfer well to other time periods. Good practice is to calibrate your model to a wide range of conditions to minimize the impacts of forcing, gage, or model physics/parameter uncertainties.

# Next up - Do it yourself!
This concludes Lesson 5. Spend some time creating your own parameter experiments.

The next lesson is [Lesson 6](Lesson-6-lakes.ipynb).

**IT IS BEST TO EITHER SHUTDOWN THIS LESSON OR CLOSE IT BEFORE PROCEEDING TO THE NEXT LESSON TO AVOID POSSIBLY EXCEEDING ALLOCATED MEMORY. Shutdown the lesson be either closing the browser tab for the lesson or selecting `Kernel -> Shut Down Kernel` in JupyterLab.**

© UCAR 2025